In [1]:
import os
from dotenv import load_dotenv, find_dotenv
from urllib.request import urlretrieve
import numpy as np
from langchain_community.embeddings import HuggingFaceBgeEmbeddings, HuggingFaceEmbeddings
from langchain_community.llms import HuggingFacePipeline
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

_= load_dotenv(find_dotenv())

## Document preparation
**We are going to download 4 publications from United States Census Bureau on the following topics:**
- Occupation, Earnings, and Job Characteristics: July 2022
- Household Income in States andMetropolitan Areas: 2022
- Poverty in States and Metropolitan Areas: 2022
- Health Insurance Coverage Status and Type by Geography: 2021 and 2022

We prepare this documents for the LLM to use as a knowledge base.

In [2]:
# Download documents from U.S. Census Bureau to local directory.
os.makedirs("us_census", exist_ok=True)
files = [
    "https://www.census.gov/content/dam/Census/library/publications/2022/demo/p70-178.pdf",
    "https://www.census.gov/content/dam/Census/library/publications/2023/acs/acsbr-017.pdf",
    "https://www.census.gov/content/dam/Census/library/publications/2023/acs/acsbr-016.pdf",
    "https://www.census.gov/content/dam/Census/library/publications/2023/acs/acsbr-015.pdf",
]
for url in files:
    file_path = os.path.join("us_census", url.rpartition("/")[2])
    urlretrieve(url, file_path)

**Split documents to smaller chunks** 

Documents should be: 
- large enough to contain enough information to answer a question, and 
- small enough to fit into the LLM prompt: Mistral-7B-v0.1 input tokens limited to 4096 tokens
- small enough to fit into the embeddings model: BAAI/bge-small-en-v1.5: input tokens limited to 512 tokens (roughly 2000 characters. Note: 1 token ~ 4 characters).

For this project, we are going to split documents to chunks of roughly 700 characters with an overlap of 50 characters.

In [3]:
# Load pdf files in the local directory
loader = PyPDFDirectoryLoader("./us_census/")

docs_before_split = loader.load()
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 700,
    chunk_overlap  = 50,
)
docs_after_split = text_splitter.split_documents(docs_before_split)

In [4]:
docs_after_split[0]

Document(metadata={'producer': 'Adobe PDF Library 16.0.5', 'creator': 'Adobe InDesign 17.1 (Windows)', 'creationdate': '2022-07-21T14:09:01-04:00', 'author': 'U.S. Census Bureau', 'moddate': '2022-07-21T14:55:54-04:00', 'subject': 'Household Economic Studies', 'title': 'Occupation, Earnings, and Job Characeristics', 'trapped': '/False', 'source': 'us_census/p70-178.pdf', 'total_pages': 21, 'page': 0, 'page_label': '1'}, page_content='Occupation, Earnings, and Job \nCharacteristics\nJuly 2022\nP70-178\nClayton Gumber and Briana Sullivan\nCurrent Population Reports\nINTRODUCTION\nWork is a critical component of our lives and provides \na way to obtain material and nonmonetary benefits \nlike employer-provided health insurance. Scholars \nsuggest that our identities are also tied to the notion \nof “what we do” (Christiansen, 1999), and that who \nwe are is determined partly by our occupational iden -\ntity (Skorikov and Vondracek, 2011). However, work \nis time consuming—the American Tim

In [5]:
avg_doc_length = lambda docs: sum([len(doc.page_content) for doc in docs])//len(docs)
avg_char_before_split = avg_doc_length(docs_before_split)
avg_char_after_split = avg_doc_length(docs_after_split)

print(f'Before split, there were {len(docs_before_split)} documents loaded, with average characters equal to {avg_char_before_split}.')
print(f'After split, there were {len(docs_after_split)} documents (chunks), with average characters equal to {avg_char_after_split} (average chunk length).')

Before split, there were 63 documents loaded, with average characters equal to 3840.
After split, there were 398 documents (chunks), with average characters equal to 624 (average chunk length).


## Text Embeddings with Hugging Face Embedding Models
At the time of writing, there are 213 text embeddings models for English on the [Massive Text Embedding Benchmark (MTEB) leaderboard](https://huggingface.co/spaces/mteb/leaderboard). For our project, we are using LangChain's HuggingFaceBgeEmbeddings (BGE models on the Hugging Face), which according to LangChain are "the best open-source embedding models". Currently, **BAAI/bge-small-en-v1.5** model is the 26th on MTEB leaderboard with max tokens: 512 tokens, embedding dimensions: 384 and model size: 0.13GB.

In [6]:
model_name = "sentence-transformers/all-mpnet-base-v2"
model_kwargs = {'device': 'cpu'}
encode_kwargs = {'normalize_embeddings': False}
huggingface_embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

/tmp/ipykernel_9025/3594361595.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  huggingface_embeddings = HuggingFaceEmbeddings(


Now we can see how a sample embedding would look like for one of those chunks.

In [7]:
sample_embedding = np.array(huggingface_embeddings.embed_query(docs_after_split[0].page_content))
print("Sample embedding of a document chunk: ", sample_embedding)
print("Size of the embedding: ", sample_embedding.shape)

Sample embedding of a document chunk:  [-4.83258255e-02  1.02571465e-01 -5.58245741e-03 -5.23291640e-02
 -4.91910009e-03  2.05073282e-02  8.31056535e-02  4.07754034e-02
 -1.21606458e-02 -2.34917155e-03  5.60068749e-02  8.79522562e-02
  1.90960541e-02 -1.30845588e-02  3.64549533e-02  1.21531012e-02
 -4.68521938e-02  5.65863252e-02 -9.13418364e-03  8.75212811e-03
 -6.85585737e-02  2.82048415e-02 -2.72464618e-04  6.39910325e-02
  5.03581436e-03 -1.46658998e-02  1.02873780e-01 -1.36262281e-02
  3.13722566e-02  7.30381459e-02  9.61405709e-02  1.98774692e-02
 -4.30643708e-02  2.22337972e-02  2.37792688e-06  2.50138324e-02
 -9.94577445e-03  8.09526537e-03 -4.37870696e-02 -3.76593955e-02
  2.09798701e-02 -3.03108152e-02 -8.07747710e-03  3.85449603e-02
  1.26188202e-02 -2.11702827e-02  9.03539825e-03  7.38854706e-03
 -4.35065329e-02 -7.12649226e-02 -1.52352387e-02  2.98430980e-03
 -1.06526300e-01 -3.26325335e-02 -9.29501560e-03  4.21158671e-02
  2.50327680e-02 -3.04142293e-02 -2.58611701e-02 -7

## Retrieval System for vector embeddings
Once we have a embedding model, we are ready to vectorize all our documents and store them in a vector store to construct a retrieval system. With specifically designed searching algorithms, a retrieval system can do similarity searching efficiently to retrieve relevant documents.

FAISS (Facebook AI Similarity Search) is a library that allows developers to quickly search for embeddings of multimedia documents that are similar to each other. It solves limitations of traditional query search engines that are optimized for hash-based searches, and provides more scalable similarity search functions (nearest-neighbor search implementations).

In [8]:
vectorstore = FAISS.from_documents(docs_after_split, huggingface_embeddings)

In [9]:
query = """What were the trends in median household income across different states in the United States between 2021 and 2022."""  # Sample question, change to other questions you are interested in.
relevant_documents = vectorstore.similarity_search(query)
print(f'There are {len(relevant_documents)} documents retrieved which are relevant to the query. Display the first one:\n')
print(relevant_documents[0].page_content)

There are 4 documents retrieved which are relevant to the query. Display the first one:

hold income in 2022 was $24,112 
(Table 1 and Figure 2). Median 
household income was lower than 
the U.S. median in 30 states and 
Puerto Rico. It was higher than the 
U.S. median in 17 states and the 
District of Columbia. The medians 
for Arizona, Oregon, and Vermont 
were not statistically different from 
the U.S. median.
From 2021 to 2022, five states—
Alabama, Alaska, Delaware, Florida, 
and Utah—showed a statistically 
significant increase in real median 
household income; 17 states 
showed a decrease. Real median 
household income in 2022 was not 
statistically different from that in 
2021 for 28 states, the District of 
Columbia, and Puerto Rico  
(Table 1).


### Create a retriever interface using vector store, we'll use it later to construct Q & A chain using LangChain.

In [10]:
# Use similarity searching algorithm and return 3 most relevant documents.
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})

**Now we have our vector store and retrieval system ready. We then need a large language model (LLM) to process information and answer the question.**

## Open-source LLMs from Hugging Face


### Hugging Face Local Pipelines

Hugging Face models can be run locally through the HuggingFacePipeline class.

In [14]:
from langchain_community.llms.huggingface_pipeline import HuggingFacePipeline

hf = HuggingFacePipeline.from_model_id(
    model_id="mistralai/Mistral-7B-v0.1",
    task="text-generation",
    pipeline_kwargs={"temperature": 0.2, "max_new_tokens": 300}
)

llm = hf

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device has 1 GPUs available. Provide device={deviceId} to `from_model_id` to use availableGPUs for execution. deviceId is -1 (default) for CPU and can be a positive integer associated with CUDA device id.
Device set to use cpu


In [16]:
llm.invoke(query)

'What were the trends in median household income across different states in the United States between 2021 and 2022.\n\n## Introduction\n\nThe median household income in the United States increased by 1.3% from 2021 to 2022, according to data from the U.S. Census Bureau. The median household income in 2022 was $67,521, up from $66,517 in 2021.\n\nThe median household income in the United States increased by 1.3% from 2021 to 2022, according to data from the U.S. Census Bureau. The median household income in 2022 was $67,521, up from $66,517 in 2021.\n\nThe median household income in the United States increased by 1.3% from 2021 to 2022, according to data from the U.S. Census Bureau. The median household income in 2022 was $67,521, up from $66,517 in 2021.\n\n## The Trends in Median Household Income\n\nThe median household income in the United States increased by 1.3% from 2021 to 2022, according to data from the U.S. Census Bureau. The median household income in 202'

## Q & A chain 
Now we have both the retrieval system for relevant documents and LLM as QA chatbot ready.

We will take our initial query, together with the relevant documents retrieved based on the results of our similarity search, to create a prompt to feed into the LLM. The LLM will take the initial query as the question and relevant documents as the context information to generate a result.

Luckily, **LangChain** provides an abstraction of the whole pipeline - **RetrievalQA**

**Let's first construct a proper prompt for our task.**

Prompt engineering is another crucial factor in LLM's performance.

In [17]:
prompt_template = """Use the following pieces of context to answer the question at the end. Please follow the following rules:
1. If you don't know the answer, don't try to make up an answer. Just say "I can't find the final answer but you may want to check the following links".
2. If you find the answer, write the answer in a concise way with five sentences maximum.

{context}

Question: {question}

Helpful Answer:
"""

PROMPT = PromptTemplate(
    template=prompt_template, input_variables=["context", "question"]
)

Call LangChain's RetrievalQA with the prompt above. 

In [18]:
retrievalQA = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": PROMPT}
)

## Use RetrievalQA invoke method to execute the chain
Note that Input of [invoke method](https://api.python.langchain.com/en/latest/chains/langchain.chains.retrieval_qa.base.RetrievalQA.html#langchain.chains.retrieval_qa.base.RetrievalQA.invoke) needs to be a dictionary.

In [19]:
# Call the QA chain with our query.
result = retrievalQA.invoke({"query": query})

In [20]:
print(result['result'])

Use the following pieces of context to answer the question at the end. Please follow the following rules:
1. If you don't know the answer, don't try to make up an answer. Just say "I can't find the final answer but you may want to check the following links".
2. If you find the answer, write the answer in a concise way with five sentences maximum.

hold income in 2022 was $24,112 
(Table 1 and Figure 2). Median 
household income was lower than 
the U.S. median in 30 states and 
Puerto Rico. It was higher than the 
U.S. median in 17 states and the 
District of Columbia. The medians 
for Arizona, Oregon, and Vermont 
were not statistically different from 
the U.S. median.
From 2021 to 2022, five states—
Alabama, Alaska, Delaware, Florida, 
and Utah—showed a statistically 
significant increase in real median 
household income; 17 states 
showed a decrease. Real median 
household income in 2022 was not 
statistically different from that in 
2021 for 28 states, the District of 
Columbia, and

In [21]:
relevant_docs = result['source_documents']
print(f'There are {len(relevant_docs)} documents retrieved which are relevant to the query.')
print("*" * 100)
for i, doc in enumerate(relevant_docs):
    print(f"Relevant Document #{i+1}:\nSource file: {doc.metadata['source']}, Page: {doc.metadata['page']}\nContent: {doc.page_content}")
    print("-"*100)

There are 3 documents retrieved which are relevant to the query.
****************************************************************************************************
Relevant Document #1:
Source file: us_census/acsbr-017.pdf, Page: 3
Content: hold income in 2022 was $24,112 
(Table 1 and Figure 2). Median 
household income was lower than 
the U.S. median in 30 states and 
Puerto Rico. It was higher than the 
U.S. median in 17 states and the 
District of Columbia. The medians 
for Arizona, Oregon, and Vermont 
were not statistically different from 
the U.S. median.
From 2021 to 2022, five states—
Alabama, Alaska, Delaware, Florida, 
and Utah—showed a statistically 
significant increase in real median 
household income; 17 states 
showed a decrease. Real median 
household income in 2022 was not 
statistically different from that in 
2021 for 28 states, the District of 
Columbia, and Puerto Rico  
(Table 1).
---------------------------------------------------------------------------------